# Dated payoff events and the stepped-bond underlying

Native products are convenient when the contract is already represented by a
library class. `ScriptedProduct` is useful when you need to describe dated
observations and payments explicitly. The payoff language is evaluated by the
native engine over simulated market paths; the Python API returns a dictionary
of path-averaged numeric variables. [Setup](../README.md).

First we write a single payment directly, then reuse the stepped note's coupon
and amortization rules. Script text is necessary for the payoff language, but
file loading and repetitive event-string assembly live in auxiliary files.

In [ ]:
import math
import pandas as pd
import quantsupport as qs
from market_data import load_market, load_pricers
from stepped_cashflows import underlying_script

market = load_market()
ref = market.quotes.reference_date
pricers = load_pricers("bond_tree.json")


## One event observes a rate and schedules a payment

An event runs on its `date`. `RateIndex` observes the specified accrual-period
rate from the model at that event; `cvg` computes a year fraction; `pays` adds
a discounted payment to a named accumulator. Here `note` is the PV accumulator,
while `rate` and `accrual` are intermediate numeric variables.

The `result_variable` names the engine's target variable; it does not filter
the returned dictionary to only that key. `evaluate_script` currently returns
values, not Python sensitivity or cashflow tables. The model is resolved from
the context configuration below, so there is no implicit zero-volatility model.

In [ ]:
fixing = ref + "1Y"
payment = fixing + "3M"
scripted_caplet = qs.ScriptedProduct.from_dict(
    {
        "name": "SCRIPTED-CAPLET",
        "currency": qs.Currency.USD,
        "result_variable": "note",
        "events": [
            {
                "date": fixing,
                "code": (
                    f'note = 0; rate = RateIndex("SOFR", "{fixing}", "{payment}"); '
                    f'accrual = cvg("{fixing}", "{payment}", "Actual360"); '
                    f'note pays 1000000 * max(rate - 0.045, 0) * accrual on "{payment}";'
                ),
            }
        ],
    }
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)
context.initialize()

with context:
    values = context.evaluate_script(
        scripted_caplet,
        n_paths=2048,
        seed=42,
    )

assert math.isfinite(values["note"]) and values["note"] > 0
pd.Series(values)


This uses the configured Hull–White model's equivalent LGM representation.
It need not exactly equal a Black caplet priced at an arbitrary smile point:
the model is fitted to an ATM calibration basket and Monte Carlo adds sampling
error. Keep model choice distinct from payoff representation.

## Reuse the stepped note's underlying payments

The contract below matches the six-year coupon and principal schedule in the
[callable-bond example](../valuation/stepped_coupon_bond.ipynb), with issuer calls omitted.
`underlying_script` translates this specific semiannual example to dated
events. It is intentionally not a replacement for the library's general
schedule construction. Its implementation is short enough to inspect in
[stepped_cashflows.py](stepped_cashflows.py).

In [ ]:
underlying = qs.SteppedCouponBond(pricing_anchor=qs.BondPricingAnchor.OAS(0.0), 
    identifier="STEPPED-UNDERLYING",
    start_date=ref,
    maturity_date=ref + "6Y",
    trade_date=ref,
    notional=1_000_000,
    currency=qs.Currency.USD,
    side=qs.Side.LongReceive,
    payment_frequency=qs.Frequency.Semiannual,
    coupon_day_counter=qs.DayCounter.Actual365,
    coupons=[
        qs.Coupon.fixed(0.045),
        qs.Coupon.floating(
            qs.MarketIndex.SOFR,
            spread=0.006,
            floor=0.0,
            cap=0.06,
        ),
    ],
    notional_percentages=[1.0, 0.90, 0.75],
    call_schedule=[],
)
script = underlying_script(underlying)
pd.DataFrame(script.to_dict()["events"]).head(3)


The first event initializes `bond`, fixes the first coupon, and schedules its
coupon plus principal reduction. Later events observe SOFR, apply spread and
cap/floor, and schedule their payments. The final payment returns all remaining
principal. The helper uses Actual/365 for the fixed coupon and SOFR's
Actual/360 for floating coupons.

This is the **non-callable underlying at zero spread**. The absence of exercise
logic is an economic difference from the callable note, not a numerical error.

In [ ]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)
context.initialize()

with context:
    stochastic = context.evaluate_script(
        script,
        n_paths=4096,
        seed=42,
    )
print(f"Calibrated scripted underlying PV: USD {stochastic['bond']:,.2f}")


## Isolate a representation check

To check that the two representations describe the same coupon and amortization
payments, use a **separate zero-volatility context**, remove issuer exercise,
and set zero Z-spread. The path count can be one because this check is
deterministic. This does not validate stochastic calibration or issuer-call
exercise, and it does not change the calibrated examples above.

The only model change is explicit below. A 0.05% PV tolerance accommodates the
native tree's time discretization for the floating coupons.

In [ ]:
deterministic_spec = pricers[0].to_dict()
deterministic_spec["models"][0]["model"]["HullWhite"]["parameter_source"] = {
    "Fixed": {"sigma": 0.0}
}
deterministic_pricer = qs.PricerConfiguration.from_dict(deterministic_spec)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    pricer_configurations=[deterministic_pricer],
)
context.initialize()

with context:
    native = context.evaluate(
        underlying.with_pricing_anchor(qs.BondPricingAnchor.z_spread(0.0)),
        [qs.Request.Value],
    )
    scripted = context.evaluate_script(
        script,
        n_paths=1,
        seed=42,
    )

assert math.isfinite(native.price) and math.isfinite(scripted["bond"])
assert math.isclose(
    native.price,
    scripted["bond"],
    rel_tol=5e-4,
    abs_tol=1e-8,
)
pd.Series(
    {
        "native_pv_usd": native.price,
        "scripted_pv_usd": scripted["bond"],
        "difference_usd": native.price - scripted["bond"],
    }
)


Keep the comparison at matching terms, spread, and model assumptions when
adapting this example. `evaluate_script` currently assembles rate models; it
does not expose arbitrary FX/equity factors, custom correlations, raw payoff
paths, or sensitivity tables through this Python entry point.